# Chapter 1 — Introduction
### Notebook 4 · Exercises

*Book reference: Section 1.5*

The book's review questions and exercises, recast as executable tasks. Every solution asserts — the assertions *are* the marking scheme.

In [1]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


In [2]:
sys.path.insert(0, str(Path.cwd()))          # so ch01_toolkit imports
import ch01_toolkit as ch1
from oe_course import ontology as ont
from oe_course.data import corpus
import pandas as pd
pd.set_option("display.width", 120)

## Review questions (§1.5)

The book asks these in prose. Answer each by producing evidence from the corpus, not by recalling a definition.

### Exercise R1 — What distinguishes a taxonomy from an ontology?

Produce two artefacts from the corpus that differ *only* in this respect, and the single metric that separates them.

In [3]:
# YOUR CODE HERE


<details>
<summary>Solution R1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [4]:
tax = ont.load_graph(corpus.get('animals-taxonomy').turtle)
onto = ont.load_graph(corpus.get('awo').turtle)
mt, mo = ont.graph_metrics(tax), ont.graph_metrics(onto)

print(f"{'metric':28s}{'taxonomy':>10s}{'ontology':>10s}")
for k in ['classes', 'subclass_axioms', 'restrictions',
          'disjointness_axioms', 'property_characteristics']:
    print(f'{k:28s}{mt[k]:>10d}{mo[k]:>10d}')

assert mt['subclass_axioms'] > 0 and mo['subclass_axioms'] > 0
assert mt['restrictions'] == 0 and mo['restrictions'] > 0
print('\nBoth have a hierarchy. Only one constrains what its terms may mean.\n'
      'The separating metric is restrictions (axioms beyond subsumption).')

metric                        taxonomy  ontology
classes                              6        16
subclass_axioms                      5        21
restrictions                         0         8
disjointness_axioms                  0         1
property_characteristics             0         2

Both have a hierarchy. Only one constrains what its terms may mean.
The separating metric is restrictions (axioms beyond subsumption).


### Exercise R2 — Give an example where an ontology changes a query answer

Show a query returning nothing on raw data and something correct after entailment. Report the recall difference numerically.

In [5]:
# YOUR CODE HERE


<details>
<summary>Solution R2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [6]:
before = len({p for p, _ in ch1.cardiac_patients(ch1.naive_union())})
after = len({p for p, _ in ch1.cardiac_patients(ch1.integrated(reason=True))})
print(f'cardiac patients found -- raw union: {before}, integrated+reasoned: {after}')
assert before == 0 and after == 4
print(f'recall {before/4:.2f} -> {after/4:.2f}')

cardiac patients found -- raw union: 0, integrated+reasoned: 4
recall 0.00 -> 1.00


### Exercise R3 — Name three properties of a 'good' ontology and test them

Choose three properties that are checkable, and evaluate the AWO against them.

In [7]:
# YOUR CODE HERE


<details>
<summary>Solution R3</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [8]:
awo = ont.load_graph(corpus.get('awo').turtle)
m = ont.graph_metrics(awo)
checks = {
    'every class is documented (labelled)':
        not any(f.smell == 'missing-label' for f in ont.scan_smells(awo)),
    'properties carry domain and range':
        not any(f.smell == 'property-without-domain-or-range'
                for f in ont.scan_smells(awo)),
    'the hierarchy is acyclic':
        not any(f.smell == 'subsumption-cycle' for f in ont.scan_smells(awo)),
}
for name, ok in checks.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
assert all(checks.values())
print('\nNote what is NOT checkable this way: whether the ontology is *true*,\n'
      'or useful for its purpose. Those need a domain expert and a use case.')

  [PASS] every class is documented (labelled)
  [PASS] properties carry domain and range
  [PASS] the hierarchy is acyclic

Note what is NOT checkable this way: whether the ontology is *true*,
or useful for its purpose. Those need a domain expert and a use case.


## Exercises (§1.5)

### Exercise E1 — Classify an unseen artefact

Write Turtle for a small artefact of your own that lands on **thesaurus** — not taxonomy, not formal ontology — and prove it.

In [9]:
my_ttl = '''
@prefix owl:  <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix ex:   <http://example.org/mine#> .
# YOUR CODE HERE
'''


<details>
<summary>Solution E1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [10]:
my_ttl = '''
@prefix owl:  <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix ex:   <http://example.org/mine#> .

ex:Instrument a owl:Class ; rdfs:label "instrument"@en .
ex:Strings    a owl:Class ; rdfs:label "strings"@en ;
              rdfs:subClassOf ex:Instrument ; skos:broader ex:Instrument .
ex:Violin     a owl:Class ; rdfs:label "violin"@en ;
              rdfs:subClassOf ex:Strings ; skos:related ex:Viola .
ex:Viola      a owl:Class ; rdfs:label "viola"@en ;
              rdfs:subClassOf ex:Strings ; skos:related ex:Violin .
'''
g = ont.load_graph(my_ttl)
result = ont.classify_spectrum(g)
print(result['level'])
for e in result['evidence']:
    print('  -', e)
assert result['level'] == 'thesaurus'
print('\nTo stay a thesaurus it must have SKOS relations and NO restrictions,\n'
      'disjointness or property characteristics. Adding one owl:disjointWith\n'
      'would reclassify it immediately.')

thesaurus
  - 4 classes, 4 labels -> named terms exist
  - 3 subsumption axioms -> a hierarchy exists
  - 3 SKOS relations -> associative/lexical structure

To stay a thesaurus it must have SKOS relations and NO restrictions,
disjointness or property characteristics. Adding one owl:disjointWith
would reclassify it immediately.


### Exercise E2 — Build a defect that no current detector catches

Construct a small ontology that is clearly badly modelled but that `scan_smells` reports as clean. This is an exercise in the **limits** of automated quality checks.

> **Hint.** Think about a relation that is *not* subsumption being modelled as subsumption.

In [11]:
# YOUR CODE HERE


<details>
<summary>Solution E2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [12]:
sneaky = '''
@prefix owl:  <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix ex:   <http://example.org/sneaky#> .

# Modelling a part-whole relation as subsumption: a wheel is NOT a kind of car.
ex:Car   a owl:Class ; rdfs:label "car"@en .
ex:Wheel a owl:Class ; rdfs:label "wheel"@en ; rdfs:subClassOf ex:Car .
ex:Door  a owl:Class ; rdfs:label "door"@en ; rdfs:subClassOf ex:Car .
ex:Car owl:disjointWith ex:Person .
ex:Person a owl:Class ; rdfs:label "person"@en .
'''
g = ont.load_graph(sneaky)
print('scanner says:', ont.smell_summary(g) or 'clean')
assert ont.smell_summary(g) == {}
print('\nYet Wheel SubClassOf Car asserts that every wheel IS a car -- the classic\n'
      'is-a / part-of confusion (Keet Ch. 6.2). No syntactic detector catches it,\n'
      'because syntactically it is a perfectly ordinary subsumption axiom.\n'
      'Catching it needs either a foundational ontology or a human. That is the\n'
      'honest boundary of the tooling in this chapter.')

scanner says: clean

Yet Wheel SubClassOf Car asserts that every wheel IS a car -- the classic
is-a / part-of confusion (Keet Ch. 6.2). No syntactic detector catches it,
because syntactically it is a perfectly ordinary subsumption axiom.
Catching it needs either a foundational ontology or a human. That is the
honest boundary of the tooling in this chapter.


## Where this chapter leaves you

You can now measure what an artefact is, show what it is worth, and detect a class of defects automatically — *and* you have seen exactly where automation stops (Exercise E2). Notebook 5 asks the next question: can an **agent** do this triage, and how would you know whether it does it well?